# 06 — Create Addresses

For every ACTIVE subscription with a successfully resolved address (from
`05_Fetch_Subscriptions.ipynb`'s Voyager lookup), `PUT`s that address onto
its target account:

```
{{host}}/rest/SubscriberService/v1/subscribers/{accountcode}
```

`addLine1` / `addLine2` / `city` / `zip` / `state` (region ISO) all come
from the Voyager `ParsedAddress_*` columns — real values, not placeholders.

Returns each new address's `id`, saved as `ship_add_id` — this is what
`07_Create_Subscription_Orders.ipynb` uses as `shipAddId` on the order.

Subscriptions whose Voyager lookup didn't resolve are skipped here (flagged
with status `"skipped"`) — they need a manual address before their order can
be created.

Inactive subscriptions don't go through this notebook at all — see
`06_Attach_Inactive_Addresses.ipynb`, which runs AFTER this one finishes
(it needs every active address here to be created first, so it can find
each account's final default service address).


## 1. Setup

In [14]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_addresses")

df_subscriptions = load_subscriptions_resolved()
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from 05_Fetch_Subscriptions.ipynb")


2026-07-29 07:04:43,135 [INFO] Loaded 264 subscriptions from 05_Fetch_Subscriptions.ipynb


In [15]:
# HARDCODED_TOKEN = "c5bf2481-f3fd-498b-a260-27c6e783782f"

# token_manager._token = HARDCODED_TOKEN
# token_manager._expires_at = datetime.now() + timedelta(hours=1)  # adjust to match the real token's actual TTL

## 2. Per-subscription address creation

In [16]:
def create_address_for_subscription(session: requests.Session, row: dict) -> dict:
    subscription_id = row["SubscriptionUSN"]
    account_number  = row["TargetAccountNumber"]

    result = {
        "SubscriptionUSN":      subscription_id,
        "TargetAccountNumber":  account_number,
        "addLine1":             row.get("ParsedAddress_addLine1"),
        "status":               "failed",
        "ship_add_id":          None,
        "error":                None,
    }

    if not row.get("ParsedAddress_parsed_ok"):
        result["status"] = "skipped"
        result["error"] = row.get("ParsedAddress_error") or "Voyager address lookup failed — needs manual address"
        return result

    status, ship_add_id, error = add_address_to_account(
        session,
        account_number,
        row["ParsedAddress_addLine1"],
        location_id=row.get("ParsedAddress_location_id") or str(subscription_id),
        address2=row.get("ParsedAddress_addLine2"),
        city=row.get("ParsedAddress_city") or "Christchurch",
        zip_code=row.get("ParsedAddress_postcode") or "1234",
        region_iso=row.get("ParsedAddress_region_iso"),
    )
    result["status"] = status
    result["ship_add_id"] = ship_add_id
    result["error"] = error

    if status == "created":
        logger.info(f"[OK] subscription {subscription_id} -> {account_number} address id={ship_add_id}")
    else:
        logger.error(f"[FAIL] subscription {subscription_id} — {error}")

    return result


## 3. Run (parallel driver)

In [17]:
def create_all_addresses(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    session = new_session(max_workers=max_workers)
    rows = df.to_dict("records")
    total = len(rows)
    results = []
    logger.info(f"Creating addresses for {total:,} subscriptions with {max_workers} workers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(create_address_for_subscription, session, row): row["SubscriptionUSN"] for row in rows}
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] == "created")
                logger.info(f"Progress: {i}/{total} — {ok} created so far")
    return pd.DataFrame(results)


df_address_results = create_all_addresses(df_subscriptions)
df_address_results.head(20)


2026-07-29 07:04:45,368 [INFO] Creating addresses for 264 subscriptions with 3 workers...
2026-07-29 07:04:51,374 [INFO] [OK] subscription V113063044 -> SR1602 address id=2219
2026-07-29 07:04:53,604 [INFO] [OK] subscription V113074082 -> SR1602 address id=2113
2026-07-29 07:04:55,939 [INFO] [OK] subscription V113082523 -> SR1602 address id=2114
2026-07-29 07:04:56,138 [INFO] [OK] subscription V113085997 -> SR1404 address id=2220
2026-07-29 07:04:58,241 [INFO] [OK] subscription V113071146 -> SR1404 address id=2115
2026-07-29 07:05:00,810 [INFO] [OK] subscription V113063150 -> SR1602 address id=2116
2026-07-29 07:05:01,210 [INFO] [OK] subscription V113070452 -> SR1404 address id=2221
2026-07-29 07:05:02,463 [INFO] [OK] subscription V113072458 -> ACCT2303 address id=2117
2026-07-29 07:05:03,492 [INFO] [OK] subscription V113067102 -> SR1404 address id=2118
2026-07-29 07:05:05,300 [INFO] [OK] subscription V113063093 -> SR1602 address id=2119
2026-07-29 07:05:05,796 [INFO] [OK] subscription

,SubscriptionUSN,TargetAccountNumber,addLine1,status,ship_add_id,error
0,V113063044,SR1602,1/4 JOHNSTON GROVE,created,2219,None
1,V113074082,SR1602,6/13 BIDDLE CRESCENT,created,2113,None
2,V113082523,SR1602,13/43 PORUTU STREET,created,2114,None
3,V113085997,SR1404,101/87 MARINE PARADE,created,2220,None
4,V113071146,SR1404,406/176 MANCHESTER STREET,created,2115,None
5,V113063150,SR1602,2/15 BIDDLE CRESCENT,created,2116,None
6,V113070452,SR1404,166A MANCHESTER STREET,created,2221,None
7,V113072458,ACCT2303,165 ENGLAND STREET,created,2117,None
8,V113067102,SR1404,502/162 MANCHESTER STREET,created,2118,None
9,V113063093,SR1602,14 OLIVER PLACE,created,2119,None


## 4. Failures / skips

In [18]:
not_created = df_address_results[df_address_results["status"] != "created"]
print(f"{len(not_created):,} / {len(df_address_results):,} addresses not created (failed or skipped)")
not_created.groupby("status").size()


33 / 264 addresses not created (failed or skipped)


status
exists     25
skipped     8
dtype: int64

## 5. Save

In [19]:
save_df("address_results", df_address_results)


Saved 264 rows -> migration_data\06_address_creation_results.csv
